# clustering — Python demo

Numerical companion to the entry [clustering](https://dictionaryofml.org/terms/clustering.html) of the [Dictionary of Applied Machine Learning](https://dictionaryofml.org/): it recomputes what the entry states and prints one line per check.

Numerical companion to the glossary entry 'clustering'. An aerial photograph of the Wachau (assets/wachau_ortho.jpg: Weissenkirchen, Rossatz, Duernstein and the Loiben slopes along the Danube bend; 7.31 x 3.65 km at 2.4 m per pixel; orthophoto (c) basemap.at, CC BY 4.0) is cut into square patches of 76 m. The parcel mask assets/wachau_labels.png (INVEKOS Schlaege 2025-1, AgrarMarkt Austria, CC BY 3.0 AT) says which patches lie mostly on vineyard parcels. These vineyard patches are the data points; the feature vector of a patch is its position (easting, northing) in km. The wine-growing areas of the Wachau are the clusters to be found: they follow the river banks and the slopes, so they are elongated and curved rather than round.

Requires NumPy and Matplotlib only, and uses fixed seeds, so the printed numbers reproduce exactly. Generated from [`pythondemos/clustering.py`](https://dictionaryofml.org/terms/clustering.py); CC BY 4.0.

In [ ]:
# Notebook shim: the script resolves output paths relative to __file__,
# which a notebook kernel does not define; everything lands in the
# working directory instead.
import os
__file__ = os.path.join(os.getcwd(), "clustering.py")
os.makedirs("pythondemos", exist_ok=True)

In [ ]:
"""Four clusterings of the vineyard patches of an aerial photograph of
the Wachau (k-means, a GMM, DBSCAN, spectral clustering), and the same
four plus convex clustering segmenting a photograph by patch color.

Purpose
-------
Numerical companion to the glossary entry 'clustering'.  An aerial
photograph of the Wachau (assets/wachau_ortho.jpg: Weissenkirchen,
Rossatz, Duernstein and the Loiben slopes along the Danube bend; 7.31 x
3.65 km at 2.4 m per pixel; orthophoto (c) basemap.at, CC BY 4.0) is cut
into square patches of 76 m.  The parcel mask assets/wachau_labels.png
(INVEKOS Schlaege 2025-1, AgrarMarkt Austria, CC BY 3.0 AT) says which
patches lie mostly on vineyard parcels.  These vineyard patches are the
data points; the feature vector of a patch is its position (easting,
northing) in km.  The wine-growing areas of the Wachau are the clusters
to be found: they follow the river banks and the slopes, so they are
elongated and curved rather than round.

The demo checks the entry's claims: DBSCAN recovers the contiguous
wine-growing areas (every DBSCAN cluster is connected by patches at
most eps apart) and marks isolated parcels as noise; k-means and the
GMM, run with the same number of clusters, split at least one area and
join patches of different areas in one cluster, since their clusters
are convex regions and ellipsoids; spectral clustering on the
k-nearest-neighbor graph of the positions yields clusters that are
connected in that graph.

The second data set is a photograph taken in the Wachau
(clustering_wachau_photo.jpg, the author's own, 640 x 480 px), cut into
10 x 10-pixel patches whose feature vectors are their mean red, green
and blue values.  Here the clusters are color classes (sky, cloud,
forest, meadow, river), and painting each patch by its cluster segments
the image.  All five methods run on these 3072 patches; convex
clustering fuses patch centroids along the edges of the 10-nearest-
neighbor graph of the colors, and the number of clusters it leaves is set
by its penalty parameter.

Deterministic: k-means and the GMM are initialized by splitting the
patches into equal parts ordered by easting (positions) or by brightness
(colors), the k-means step of spectral clustering by farthest-first
seeding from the first patch (no randomness).  Self-contained: numpy +
matplotlib only.

Blocks
------
[B-patches]  Cut the photograph into 32x32-pixel patches (76 m), find the
             patches that lie mostly on vineyard parcels, and take their
             positions in km as feature vectors; check the counts.
[B-dbscan]   DBSCAN with eps = 1.5 patch spacings and 4 points: check
             that every cluster is eps-connected, that noise points have
             fewer than 4 neighbors, and count the areas with at least
             20 patches (the number of clusters handed to the others).
[B-kmeans]   k-means on the positions: check that it splits at least one
             DBSCAN area and joins patches of different areas.
[B-gmm]      A GMM on the positions, hard-assigned by the largest degree
             of belonging: the same two checks.
[B-spectral] Spectral clustering on the 10-nearest-neighbor graph of the
             positions: the eigenvectors of the normalized Laplacian
             matrix for the k smallest eigenvalues, rows normalized to
             unit length, and k-means on them (Ng, Jordan and Weiss).
             Check that the Laplacian matrix is positive semi-definite
             with one zero eigenvalue per connected component of the
             graph and that every spectral cluster is connected in
             the graph.
[B-images]   The photograph at patch resolution with the vineyard patches
             marked, and one copy per method with the vineyard patches
             painted by cluster (DBSCAN noise in black).
[B-photo]    The photograph cut into 48 x 64 patches with mean-RGB feature
             vectors; k-means with k = 5 and its clustering error for
             k = 1, ..., 10 (the elbow curve), a GMM, DBSCAN in color
             space, spectral clustering on the 10-nearest-neighbor graph
             of the colors, and convex clustering with that graph's edge
             weights along a path of penalty parameters; one painted copy
             per method, each patch in the mean color of its cluster with
             white lines between clusters (DBSCAN noise in black).
[B-plot]     Write the positions of the vineyard patches for the entry's
             scatter, plus the preview.

Outputs
-------
clustering_patches.csv : x, y, share -- every patch: position in km and
                         its vineyard share
clustering_points.csv  : x1, x2 -- positions of the vineyard patches
clustering_wachau_original.png : the photograph at patch resolution
clustering_wachau_vineyard.png : the same with the vineyard patches marked
clustering_wachau_kmeans.png, _gmm.png, _dbscan.png, _spectral.png :
                         the vineyard patches painted by cluster
clustering_wachau_photo.jpg : input -- the photograph (640 x 480 px)
clustering_elbow.csv   : k, error -- k-means clustering error of the photo
                         patches for k = 1, ..., 10
clustering_photo_kmeans.png, _gmm.png, _dbscan.png, _spectral.png,
_convex.png            : the photo patches painted by cluster
clustering.png : preview (checking only) -- top: the scatter, the four
                 painted vineyard copies and the elbow curve; bottom: the
                 photograph and its five segmentations
"""

from pathlib import Path

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.image import imread

OUT_DIR = Path(__file__).parent

report = []                         # collects (check name, pass/fail) pairs


def check(name, ok):                # records and prints one verification
    report.append((name, bool(ok)))
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")

**[B-patches]** Cut the photograph into 32x32-pixel patches (76 m), find the patches that lie mostly on vineyard parcels, and take their positions in km as feature vectors; check the counts.

In [ ]:
PATCH = 32
WIDTH_KM = 7.31                     # the strip is 7.31 km wide
photo = imread(OUT_DIR.parent / "assets" / "wachau_ortho.jpg") / 255.0
labels = np.asarray(imread(OUT_DIR.parent / "assets" / "wachau_labels.png"))
if labels.ndim == 3:
    labels = labels[:, :, 0]
if labels.max() <= 1.0 and labels.dtype != np.uint8:
    labels = np.rint(labels * 255).astype(np.uint8)
rows = photo.shape[0] // PATCH
cols = photo.shape[1] // PATCH
tiles = photo[:rows * PATCH, :cols * PATCH].reshape(
    rows, PATCH, cols, PATCH, 3).mean(axis=(1, 3))
share = (labels[:rows * PATCH, :cols * PATCH] == 1).reshape(
    rows, PATCH, cols, PATCH).mean(axis=(1, 3))
spacing = PATCH * WIDTH_KM / photo.shape[1]           # km between patches
grid_r, grid_c = np.divmod(np.arange(rows * cols), cols)
positions = np.column_stack([grid_c * spacing, (rows - 1 - grid_r) * spacing])
with open(OUT_DIR / "clustering_patches.csv", "w") as f:
    f.write("x,y,share\n")
    for (a, b), s in zip(positions, share.ravel()):
        f.write(f"{a:.4f},{b:.4f},{s:.3f}\n")
is_vine = share.ravel() > 0.5
X = positions[is_vine]                                # the data points
vine_r, vine_c = grid_r[is_vine], grid_c[is_vine]
print(f"  photograph cut into {rows} x {cols} = {rows * cols} patches of "
      f"{PATCH} px = {1000 * spacing:.0f} m; {len(X)} lie mostly on vineyard "
      f"parcels")
check("[B-patches] every vineyard patch has a two-number feature vector",
      X.shape == (is_vine.sum(), 2) and 900 < len(X) < 1100)

**[B-dbscan]** DBSCAN with eps = 1.5 patch spacings and 4 points: check that every cluster is eps-connected, that noise points have fewer than 4 neighbors, and count the areas with at least 20 patches (the number of clusters handed to the others).

In [ ]:
EPS = 1.5 * spacing
MIN_PTS = 4
dist = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=2))
within = dist <= EPS
core = within.sum(axis=1) >= MIN_PTS               # counting the point itself


def dbscan(within, core):
    """Density-connected components of the core points; -1 marks noise."""
    lab = -np.ones(len(core), dtype=int)
    current = 0
    for seed in range(len(core)):
        if lab[seed] != -1 or not core[seed]:
            continue
        lab[seed] = current
        stack = [seed]
        while stack:
            u = stack.pop()
            if not core[u]:
                continue
            for v in np.flatnonzero(within[u]):
                if lab[v] == -1:
                    lab[v] = current
                    stack.append(v)
        current += 1
    return lab


lab_db = dbscan(within, core)
n_db = lab_db.max() + 1
sizes_db = np.bincount(lab_db[lab_db >= 0])
areas = [c for c in range(n_db) if sizes_db[c] >= 20]
NRCLUSTER = len(areas)
print(f"  DBSCAN: {n_db} clusters of sizes {sorted(sizes_db.tolist(), reverse=True)}, "
      f"{int((lab_db < 0).sum())} noise patches; {NRCLUSTER} areas with at "
      f"least 20 patches")


def components(mask, adjacency):
    """Number of connected components of the subgraph induced by mask."""
    seen = np.zeros(len(mask), dtype=bool)
    count = 0
    for s in np.flatnonzero(mask):
        if seen[s]:
            continue
        count += 1
        stack = [s]
        seen[s] = True
        while stack:
            u = stack.pop()
            for v in np.flatnonzero(adjacency[u] & mask & ~seen):
                seen[v] = True
                stack.append(v)
    return count


check("[B-dbscan] every DBSCAN cluster is connected by patches at most eps apart",
      all(components(lab_db == c, within) == 1 for c in range(n_db)))
check("[B-dbscan] every noise patch has fewer than 4 patches within eps",
      bool(np.all(within[lab_db < 0].sum(axis=1) < MIN_PTS)))
check("[B-dbscan] the areas with at least 20 patches are the wine-growing "
      "areas (between 4 and 8 of them)", 4 <= NRCLUSTER <= 8)

**[B-kmeans]** k-means on the positions: check that it splits at least one DBSCAN area and joins patches of different areas.

In [ ]:
def assign(A, cents):
    return ((A[:, None, :] - cents) ** 2).sum(axis=2).argmin(axis=1)


def lloyd(A, cents):
    lab = assign(A, cents)
    for _ in range(300):
        new = np.stack([A[lab == c].mean(axis=0) if (lab == c).any()
                        else cents[c] for c in range(len(cents))])
        new_lab = assign(A, new)
        if np.array_equal(new_lab, lab) and np.allclose(new, cents):
            return new_lab, new
        lab, cents = new_lab, new
    return lab, cents


def start_from_easting(A, k):
    parts = np.array_split(np.argsort(A[:, 0]), k)
    return np.stack([A[g].mean(axis=0) for g in parts])


lab_km, cent_km = lloyd(X, start_from_easting(X, NRCLUSTER))


def splits_and_joins(lab, areas, lab_db):
    """How a hard clustering treats the DBSCAN areas.

    split: an area whose patches fall into more than one cluster (at
    least 20 patches on each side); join: a cluster that holds at least
    20 patches of two different areas.
    """
    split = 0
    for a in areas:
        counts = np.bincount(lab[lab_db == a], minlength=lab.max() + 1)
        split += int((counts >= 20).sum() >= 2)
    join = 0
    for c in range(lab.max() + 1):
        counts = [int(((lab == c) & (lab_db == a)).sum()) for a in areas]
        join += int(sum(n >= 20 for n in counts) >= 2)
    return split, join


split_km, join_km = splits_and_joins(lab_km, areas, lab_db)
print(f"  k-means with k = {NRCLUSTER}: splits {split_km} area(s), "
      f"{join_km} cluster(s) join two areas")
check("[B-kmeans] k-means splits at least one wine-growing area", split_km >= 1)
check("[B-kmeans] at least one k-means cluster joins two areas", join_km >= 1)

**[B-gmm]** A GMM on the positions, hard-assigned by the largest degree of belonging: the same two checks.

In [ ]:
def normal_pdf(A, mean, cov):
    d = A.shape[1]
    diff = A - mean
    quad = np.einsum("ij,jk,ik->i", diff, np.linalg.inv(cov), diff)
    return np.exp(-0.5 * quad) / np.sqrt(((2.0 * np.pi) ** d)
                                         * np.linalg.det(cov))


parts = np.array_split(np.argsort(X[:, 0]), NRCLUSTER)
means = np.stack([X[g].mean(axis=0) for g in parts])
covs = np.stack([np.cov(X[g].T) + 1e-6 * np.eye(2) for g in parts])
p = np.full(NRCLUSTER, 1.0 / NRCLUSTER)
for _ in range(300):
    joint = np.stack([p[c] * normal_pdf(X, means[c], covs[c])
                      for c in range(NRCLUSTER)])
    posterior = joint / joint.sum(axis=0)
    weight = posterior.sum(axis=1)
    p = weight / len(X)
    means = (posterior @ X) / weight[:, None]
    covs = np.stack([
        (posterior[c][:, None] * (X - means[c])).T @ (X - means[c]) / weight[c]
        + 1e-6 * np.eye(2) for c in range(NRCLUSTER)])
lab_gmm = posterior.argmax(axis=0)
split_gmm, join_gmm = splits_and_joins(lab_gmm, areas, lab_db)
print(f"  GMM with k = {NRCLUSTER}: splits {split_gmm} area(s), "
      f"{join_gmm} cluster(s) join two areas; {(posterior.max(axis=0) < 0.8).mean():.0%} "
      f"of the patches graded below 0.8")
check("[B-gmm] the degrees of belonging of each patch sum to one",
      np.allclose(posterior.sum(axis=0), 1.0))
check("[B-gmm] the GMM splits at least one wine-growing area", split_gmm >= 1)
check("[B-gmm] at least one GMM cluster joins two areas", join_gmm >= 1)

**[B-spectral]** Spectral clustering on the 10-nearest-neighbor graph of the positions: the eigenvectors of the normalized Laplacian matrix for the k smallest eigenvalues, rows normalized to unit length, and k-means on them (Ng, Jordan and Weiss). Check that the Laplacian matrix is positive semi-definite with one zero eigenvalue per connected component of the graph and that every spectral cluster is connected in the graph.

In [ ]:
NEIGHBORS = 10
d_off = dist.copy()
np.fill_diagonal(d_off, np.inf)
nearest = np.argsort(d_off, axis=1)[:, :NEIGHBORS]
adj = np.zeros_like(within)
np.put_along_axis(adj, nearest, True, axis=1)
adj = adj | adj.T                                  # edges in both directions
bandwidth = np.median(np.take_along_axis(d_off, nearest, axis=1))
weights = np.where(adj, np.exp(-d_off ** 2 / (2.0 * bandwidth ** 2)), 0.0)
degree = weights.sum(axis=1)
laplacian = np.diag(degree) - weights
scale = 1.0 / np.sqrt(degree)
normalized = np.eye(len(X)) - scale[:, None] * weights * scale[None, :]
eigval, eigvec = np.linalg.eigh(laplacian)
eigval_n, eigvec_n = np.linalg.eigh(normalized)
n_comp = int((eigval < 1e-9).sum())               # connected components
embedding = eigvec_n[:, :NRCLUSTER]                # new feature vectors
embedding /= np.linalg.norm(embedding, axis=1, keepdims=True)


def farthest_first(A, k):
    """Deterministic start: the first point, then always the farthest one."""
    chosen = [0]
    for _ in range(k - 1):
        d2 = ((A[:, None, :] - A[chosen]) ** 2).sum(axis=2).min(axis=1)
        chosen.append(int(d2.argmax()))
    return A[chosen]


lab_sp, _ = lloyd(embedding, farthest_first(embedding, NRCLUSTER))
split_sp, join_sp = splits_and_joins(lab_sp, areas, lab_db)
print(f"  spectral on the {NEIGHBORS}-NN graph ({int(adj.sum() // 2)} edges, "
      f"{n_comp} connected component(s)): splits {split_sp} area(s), "
      f"{join_sp} cluster(s) join two areas")
check("[B-spectral] the Laplacian matrix is positive semi-definite with one "
      "zero eigenvalue per connected component",
      eigval.min() > -1e-9 and n_comp == components(np.ones(len(X), bool), adj))
check("[B-spectral] every spectral cluster is connected in the graph",
      all(components(lab_sp == c, adj) == 1 for c in range(NRCLUSTER)))

**[B-images]** The photograph at patch resolution with the vineyard patches marked, and one copy per method with the vineyard patches painted by cluster (DBSCAN noise in black).

In [ ]:
def save_image(arr, name, zoom=3):
    img = np.clip(arr, 0.0, 1.0).repeat(zoom, axis=0).repeat(zoom, axis=1)
    plt.imsave(OUT_DIR / name, img)


PALETTE = np.array([[0.95, 0.75, 0.10], [0.15, 0.35, 0.85], [0.05, 0.75, 0.85],
                    [0.60, 0.10, 0.10], [0.15, 0.65, 0.25], [0.95, 0.55, 0.75],
                    [0.55, 0.30, 0.05], [0.45, 0.45, 0.45], [0.65, 0.15, 0.75],
                    [0.55, 0.85, 0.30]])   # ordered by distinct brightness
save_image(tiles, "clustering_wachau_original.png")
dimmed = 0.35 * tiles + 0.35
marked = dimmed.copy()
marked[vine_r, vine_c] = tiles[vine_r, vine_c]
save_image(marked, "clustering_wachau_vineyard.png")


def paint(lab, name):
    img = dimmed.copy()
    for i, (r, c) in enumerate(zip(vine_r, vine_c)):
        img[r, c] = PALETTE[lab[i] % len(PALETTE)] if lab[i] >= 0 else 0.0
    save_image(img, name)
    return img


painted = {m: paint(lab, f"clustering_wachau_{m}.png")
           for m, lab in (("kmeans", lab_km), ("gmm", lab_gmm),
                          ("dbscan", lab_db), ("spectral", lab_sp))}
check("[B-images] one painted copy per method",
      all((OUT_DIR / f"clustering_wachau_{m}.png").exists() for m in painted))

**[B-photo]** The photograph cut into 48 x 64 patches with mean-RGB feature vectors; k-means with k = 5 and its clustering error for k = 1, ..., 10 (the elbow curve), a GMM, DBSCAN in color space, spectral clustering on the 10-nearest-neighbor graph of the colors, and convex clustering with that graph's edge weights along a path of penalty parameters; one painted copy per method, each patch in the mean color of its cluster with white lines between clusters (DBSCAN noise in black).

In [ ]:
PHOTO_PATCH = 10
snap = np.asarray(imread(OUT_DIR / "clustering_wachau_photo.jpg"),
                  dtype=float) / 255.0
p_rows, p_cols = snap.shape[0] // PHOTO_PATCH, snap.shape[1] // PHOTO_PATCH
colors = snap[:p_rows * PHOTO_PATCH, :p_cols * PHOTO_PATCH].reshape(
    p_rows, PHOTO_PATCH, p_cols, PHOTO_PATCH, 3).mean(axis=(1, 3))
C = colors.reshape(-1, 3)                       # feature vectors: mean RGB
K_PHOTO = 5
print(f"  photograph cut into {p_rows} x {p_cols} = {len(C)} patches of "
      f"{PHOTO_PATCH} px; feature vector = mean (R, G, B) in [0, 1]")


def brightness_start(A, k):
    """Deterministic start: k equal parts of the points ordered by brightness."""
    parts = np.array_split(np.argsort(A.sum(axis=1)), k)
    return np.stack([A[g].mean(axis=0) for g in parts])


# k-means with k = 5, and the clustering error for k = 1, ..., 10
lab_km_p, cents_p = lloyd(C, brightness_start(C, K_PHOTO))
elbow = []
for k in range(1, 11):
    lab_k, cents_k = lloyd(C, brightness_start(C, k))
    elbow.append(float(((C - cents_k[lab_k]) ** 2).sum(axis=1).mean()))
with open(OUT_DIR / "clustering_elbow.csv", "w") as f:
    f.write("k,error\n")
    for k, e in enumerate(elbow, start=1):
        f.write(f"{k},{e:.6f}\n")
bright = cents_p.mean(axis=1)
print(f"  k-means with k = {K_PHOTO}: cluster sizes {np.bincount(lab_km_p).tolist()}, "
      f"centroid brightness {np.round(np.sort(bright), 2).tolist()}; clustering "
      f"error for k = 1..10: {[round(e, 4) for e in elbow]}")
check("[B-photo] every patch of the photograph has a three-number feature vector",
      C.shape == (p_rows * p_cols, 3) and len(C) > 2000)
check("[B-photo] the k-means clustering error falls from k = 1 to 5 to 10",
      elbow[0] > elbow[4] > elbow[9])
check("[B-photo] k-means separates a bright (sky) from a dark (forest) cluster",
      bright.max() - bright.min() > 0.4)


def em_gmm(A, k, iters=300):
    """EM for a GMM with k components; returns hard labels and posteriors."""
    d = A.shape[1]
    parts = np.array_split(np.argsort(A.sum(axis=1)), k)
    mu = np.stack([A[g].mean(axis=0) for g in parts])
    sig = np.stack([np.cov(A[g].T) + 1e-5 * np.eye(d) for g in parts])
    pk = np.full(k, 1.0 / k)
    for _ in range(iters):
        joint = np.stack([pk[c] * normal_pdf(A, mu[c], sig[c])
                          for c in range(k)])
        post = joint / np.maximum(joint.sum(axis=0), 1e-300)
        wgt = post.sum(axis=1)
        pk = wgt / len(A)
        mu = (post @ A) / wgt[:, None]
        sig = np.stack([(post[c][:, None] * (A - mu[c])).T @ (A - mu[c])
                        / wgt[c] + 1e-5 * np.eye(d) for c in range(k)])
    return post.argmax(axis=0), post


lab_gmm_p, post_p = em_gmm(C, K_PHOTO)
print(f"  GMM with k = {K_PHOTO}: cluster sizes {np.bincount(lab_gmm_p).tolist()}; "
      f"{(post_p.max(axis=0) < 0.8).mean():.0%} of the patches graded below 0.8")
check("[B-photo] the degrees of belonging of each photo patch sum to one",
      np.allclose(post_p.sum(axis=0), 1.0))

# DBSCAN in color space
dist_p = np.sqrt(((C[:, None, :] - C[None, :, :]) ** 2).sum(axis=2))
EPS_PHOTO, MIN_PTS_PHOTO = 0.03, 8
within_p = dist_p <= EPS_PHOTO
lab_db_p = dbscan(within_p, within_p.sum(axis=1) >= MIN_PTS_PHOTO)
n_db_p = lab_db_p.max() + 1
sizes_db_p = np.bincount(lab_db_p[lab_db_p >= 0])
noise_p = float((lab_db_p < 0).mean())
print(f"  DBSCAN with eps = {EPS_PHOTO}, minPts = {MIN_PTS_PHOTO}: {n_db_p} clusters "
      f"of sizes {sorted(sizes_db_p.tolist(), reverse=True)}, {noise_p:.0%} noise")
check("[B-photo] DBSCAN chains through the gradual color transitions: its two "
      "largest clusters hold over 80% of the patches, 1-30% are noise",
      np.sort(sizes_db_p)[-2:].sum() > 0.8 * len(C) and 0.01 <= noise_p <= 0.30)


def knn_graph(D, neighbors):
    """Symmetrized k-nearest-neighbor graph with Gaussian edge weights."""
    off = D.copy()
    np.fill_diagonal(off, np.inf)
    near = np.argsort(off, axis=1)[:, :neighbors]
    A = np.zeros(D.shape, dtype=bool)
    np.put_along_axis(A, near, True, axis=1)
    A = A | A.T
    bw = np.median(np.take_along_axis(off, near, axis=1))
    return A, np.where(A, np.exp(-off ** 2 / (2.0 * bw ** 2)), 0.0)


# spectral clustering on the 10-nearest-neighbor graph of the colors
adj_p, weights_p = knn_graph(dist_p, NEIGHBORS)
deg_p = weights_p.sum(axis=1)
sc_p = 1.0 / np.sqrt(deg_p)
norm_lap_p = np.eye(len(C)) - sc_p[:, None] * weights_p * sc_p[None, :]
_, vec_p = np.linalg.eigh(norm_lap_p)
emb_p = vec_p[:, :K_PHOTO]
emb_p /= np.linalg.norm(emb_p, axis=1, keepdims=True)
lab_sp_p, _ = lloyd(emb_p, farthest_first(emb_p, K_PHOTO))
n_comp_p = components(np.ones(len(C), bool), adj_p)
connected_sp = sum(components(lab_sp_p == c, adj_p) == 1 for c in range(K_PHOTO))
print(f"  spectral on the {NEIGHBORS}-NN color graph ({int(adj_p.sum() // 2)} edges, "
      f"{n_comp_p} connected component(s)): cluster sizes "
      f"{np.bincount(lab_sp_p).tolist()}, {connected_sp} of {K_PHOTO} clusters "
      f"connected in the graph")
check("[B-photo] the spectral feature vectors have k entries, one row per patch",
      emb_p.shape == (len(C), K_PHOTO))


def label_components(adjacency):
    """Connected-component label of every node."""
    lab = -np.ones(len(adjacency), dtype=int)
    current = 0
    for s in range(len(adjacency)):
        if lab[s] >= 0:
            continue
        lab[s] = current
        stack = [s]
        while stack:
            u = stack.pop()
            for v in np.flatnonzero(adjacency[u] & (lab < 0)):
                lab[v] = current
                stack.append(v)
        current += 1
    return lab


def convex_clustering(A, adjacency, W, alphas, rho=1.0, iters=200):
    """Minimize sum_r ||x_r - w_r||^2 + alpha sum_edges W_rr' ||w_r - w_r'||
    by alternating updates with one splitting variable per edge (Chi and
    Lange, 2015), warm-started along the path of alphas.  Returns, per
    alpha, the labels of the clusters of patches that share a centroid."""
    i_idx, j_idx = np.nonzero(np.triu(adjacency, 1))
    a_edge = W[i_idx, j_idx]
    m = len(A)
    lap = np.diag((np.bincount(i_idx, minlength=m)
                   + np.bincount(j_idx, minlength=m)).astype(float))
    lap[i_idx, j_idx] -= 1.0
    lap[j_idx, i_idx] -= 1.0                                  # E^T E
    M = np.linalg.inv(2.0 * np.eye(m) + rho * lap)
    V = A[i_idx] - A[j_idx]
    U = np.zeros_like(V)
    out = []
    for alpha in alphas:
        for _ in range(iters):
            R = np.zeros_like(A)
            np.add.at(R, i_idx, V - U)
            np.add.at(R, j_idx, U - V)
            Wc = M @ (2.0 * A + rho * R)                      # centroid update
            Z = Wc[i_idx] - Wc[j_idx] + U
            nz = np.maximum(np.linalg.norm(Z, axis=1), 1e-12)
            V = np.maximum(0.0, 1.0 - alpha * a_edge / (rho * nz))[:, None] * Z
            U = U + Wc[i_idx] - Wc[j_idx] - V
        fused = np.zeros((m, m), dtype=bool)
        keep = np.linalg.norm(V, axis=1) == 0.0                # fused edges
        fused[i_idx[keep], j_idx[keep]] = True
        fused |= fused.T
        out.append((float(alpha), label_components(fused)))
    return out


ALPHAS = np.geomspace(0.05, 500.0, 17)
path = convex_clustering(C, adj_p, weights_p, ALPHAS)
totals = [int(lab.max() + 1) for _, lab in path]
big = [int((np.bincount(lab) >= 20).sum()) for _, lab in path]
# the point on the path where about k clusters of 20 or more patches remain;
# the rest are small clusters of a few patches each (ties: the larger alpha)
alpha_cc, lab_cc_p = min(path, key=lambda t: (
    abs(int((np.bincount(t[1]) >= 20).sum()) - K_PHOTO), -t[0]))
sizes_cc = sorted(np.bincount(lab_cc_p).tolist(), reverse=True)
print("  convex clustering path (alpha: clusters / clusters of 20+ patches): "
      + ", ".join(f"{a:.3g}: {t}/{b}" for (a, _), t, b in zip(path, totals, big)))
print(f"  chosen alpha = {alpha_cc:.3g}: {len(sizes_cc)} clusters of sizes {sizes_cc}")
check("[B-photo] the number of convex-clustering clusters falls along the path "
      "from about a thousand to a handful",
      totals[0] > 500 and totals[-1] <= 10 and all(
          a >= b for a, b in zip(totals, totals[1:])))
check("[B-photo] a penalty parameter on the path leaves about k clusters of 20 "
      "or more patches",
      abs(sum(sz >= 20 for sz in sizes_cc) - K_PHOTO) <= 1)


def paint_photo(lab, name, zoom=8):
    """Each patch in the mean color of its cluster, white lines between
    clusters, DBSCAN noise in black."""
    lab2 = lab.reshape(p_rows, p_cols)
    img = np.zeros((p_rows, p_cols, 3))
    for c in np.unique(lab):
        img[lab2 == c] = C[lab == c].mean(axis=0) if c >= 0 else 0.0
    big = img.repeat(zoom, axis=0).repeat(zoom, axis=1)
    above = np.zeros((p_rows, p_cols), dtype=bool)
    above[1:] = lab2[1:] != lab2[:-1]
    left = np.zeros((p_rows, p_cols), dtype=bool)
    left[:, 1:] = lab2[:, 1:] != lab2[:, :-1]
    for r, c in zip(*np.nonzero(above)):
        big[r * zoom, c * zoom:(c + 1) * zoom] = 1.0
    for r, c in zip(*np.nonzero(left)):
        big[r * zoom:(r + 1) * zoom, c * zoom] = 1.0
    plt.imsave(OUT_DIR / name, np.clip(big, 0.0, 1.0))
    return big


painted_photo = {m: paint_photo(lab, f"clustering_photo_{m}.png")
                 for m, lab in (("kmeans", lab_km_p), ("gmm", lab_gmm_p),
                                ("dbscan", lab_db_p), ("spectral", lab_sp_p),
                                ("convex", lab_cc_p))}
check("[B-photo] one painted copy of the photograph per method",
      all((OUT_DIR / f"clustering_photo_{m}.png").exists() for m in painted_photo))

**[B-plot]** Write the positions of the vineyard patches for the entry's scatter, plus the preview.

In [ ]:
with open(OUT_DIR / "clustering_points.csv", "w") as f:
    f.write("x1,x2\n")
    for a, b in X:
        f.write(f"{a:.4f},{b:.4f}\n")

fig, grid = plt.subplots(2, 6, figsize=(23, 7.2))
axes = grid[0]
ax = axes[0]
ax.plot(X[:, 0], X[:, 1], "o", color="0.4", markersize=2.0, linestyle="none",
        label="vineyard patch")
ax.set_aspect("equal")
ax.set_xlabel("easting (km)")
ax.set_ylabel("northing (km)")
ax.set_title("vineyard patches of the Wachau strip")
ax.legend(frameon=False, fontsize=8, loc="lower right")
for ax, m, title in zip(axes[1:5], ("kmeans", "gmm", "dbscan", "spectral"),
                        ("k-means", "GMM", "DBSCAN", "spectral")):
    ax.imshow(np.clip(painted[m], 0, 1))
    ax.set_title(title)
    ax.set_xlabel("patch column")
    ax.set_ylabel("patch row")
ax = axes[5]
ax.plot(range(1, 11), elbow, "o-", color="0.2", label="k-means on the photo")
ax.set_xlabel("number k of clusters")
ax.set_ylabel("clustering error")
ax.set_title("elbow curve of the photo patches")
ax.legend(frameon=False, fontsize=8)
axes = grid[1]
axes[0].imshow(snap)
axes[0].set_title("photograph (Wachau)")
for ax, m, title in zip(axes[1:], ("kmeans", "gmm", "dbscan", "spectral", "convex"),
                        ("k-means", "GMM", "DBSCAN (noise black)", "spectral",
                         "convex clustering")):
    ax.imshow(np.clip(painted_photo[m], 0, 1))
    ax.set_title(title)
for ax in axes:
    ax.set_xlabel("pixel column")
    ax.set_ylabel("pixel row")
fig.tight_layout()
fig.savefig(OUT_DIR / "clustering.png", dpi=150)
plt.close(fig)
check("[B-plot] the scatter file was written",
      (OUT_DIR / "clustering_points.csv").exists())

passed = sum(1 for _, ok in report if ok)
print(f"\n{passed}/{len(report)} checks pass")